In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

ROOT = Path("..")
sys.path.append(str(ROOT / "src"))
from generar_datos import generar_dataset

METRICAS = ["PrecisionPase", "VelReaccion", "Efectividad1v1", "VelSprint",
            "DistAltaInt", "AciertoRegate", "Recuperaciones", "TomaDecision"]

def pipe_logit():
    return Pipeline([("sc", StandardScaler()),
                     ("m", LogisticRegression(max_iter=2000))])

def pipe_rf():
    return Pipeline([("sc", StandardScaler()),
                     ("m", RandomForestClassifier(n_estimators=300, random_state=7))])

TAMANOS = [40, 80, 160, 320, 640, 1280]
SEMILLAS = range(5)   # 5 réplicas por tamaño para promediar la suerte de la simulación

filas = []
for n in TAMANOS:
    for seed in SEMILLAS:
        df = generar_dataset(n, seed=seed)
        X, y = df[METRICAS], df["AltoPotencial"]
        cv = StratifiedKFold(5, shuffle=True, random_state=7)
        for nombre, ctor in [("Logística", pipe_logit), ("Random Forest", pipe_rf)]:
            auc = cross_val_score(ctor(), X, y, cv=cv, scoring="roc_auc").mean()
            filas.append({"n": n, "seed": seed, "modelo": nombre, "auc": auc})

res = pd.DataFrame(filas)
resumen = res.groupby(["modelo", "n"])["auc"].agg(["mean", "std"]).round(3)
print(resumen)

                     mean    std
modelo        n                 
Logística     40    0.765  0.133
              80    0.822  0.046
              160   0.842  0.022
              320   0.845  0.027
              640   0.848  0.016
              1280  0.859  0.007
Random Forest 40    0.750  0.134
              80    0.714  0.081
              160   0.777  0.026
              320   0.798  0.020
              640   0.805  0.018
              1280  0.830  0.006
